# BDB27 — fallback validation: is the shallow break a measurable capacity the battery misses?

**What this notebook is.** The outcome route is capped: the tracking on this machine carries the
draft-prospect cohort only, so a better match cannot manufacture opponents the file does not have.
This notebook tests two quantities that need **no opponent identifier at all**, plus one
descriptive statement, and applies a verdict rule that was written down before any of the numbers
were computed (`notes/fallback-validation-spec.md`, committed at `c95c983`).

1. **Leg 1 — reliability (spec §1).** Is 30–60° `apex_retention` a stable player trait? ICC(1) with a
   player-level bootstrap, split-half (odd/even) with Spearman–Brown, and a 2024↔2025 within-player
   correlation. A **proxy**: it measures the reliability of *in-game band retention*, not of a drill
   that does not exist.
2. **Leg 2 — discriminant validity (spec §2).** Does the existing Combine battery already reach the
   band? Per drill, the player's shallow-break mean (S) and deep-plant mean (P) regressed on the
   standardised drill, and the **paired** difference Δ = b_P − b_S with its own interval.
3. **Leg 3 — speed coverage (spec §3).** Descriptive only: what entry speed the Combine ever records,
   against what the band requires. No criterion, no verdict, no inference.
4. **The verdict (spec §4).** Six rules, evaluated in order; the first that fires is the headline, and
   every later rule still prints.

**Two things this notebook will not do.** It will not stop on a number: every threshold prints a
label with its denominator and the run continues on whatever the data contains (D202). And it will
not upgrade a null into a claim: a null on the 3-cone or the short shuttle — fitted on 23 and 31
players — is written "not detectable", never "not present".

**This is a diagnostic note, not a deliverable.** Nothing here enters `TASK.md`, the band table, the
four figures or the report draft. Every number is `[FULL]`.

**To run it: Run All. Nothing below §1 needs editing.** No CLI, no arguments, no installs. If an
input cannot be found, the notebook names every path it tried and what to set in §1.

## 1. Paths — the only cell to edit

* `GAME_TURNS` — the FULL DB turn table written by the full run.
  `None` = **search** the repo, `data/` and the usual home folders for `bdb27_full_db_turns.parquet`.
* `COMBINE_TURNS` — the FULL Combine control pass (D190).
  `None` = search for `combine_db_turns_floor2.5_control.csv`.
* `COMBINE_RESULTS` — `data/raw/combine_results.csv`, joined on `nfl_id` where
  `combine_position == 'DB'`.
* `OUT_DIR` — where the five artefacts are written. `None` = `outputs/fallback_validation`.
* `N_BOOTSTRAP` — replications for the player-level bootstraps. The spec fixes 2,000; lower it only
  to make a laptop quick, and say so when quoting a number.
* `STAMP` — the provenance stamp printed beside every number. `FULL` for the committed data.

If a file is missing the notebook prints every path it tried. That is one of the only three
legitimate failures in this design; a small or unusual *number* is never one of them.

In [ ]:
# ======================= CONFIG — the only cell to edit =========================
GAME_TURNS = None        # None = search for bdb27_full_db_turns.parquet
COMBINE_TURNS = None     # None = search for combine_db_turns_floor2.5_control.csv
COMBINE_RESULTS = None   # None = search for combine_results.csv
OUT_DIR = None           # None = <repo>/outputs/fallback_validation
N_BOOTSTRAP = 2000       # player-level bootstrap replications (spec fixes 2,000)
# Provenance stamp, defined here so no cell order can change it.
STAMP = "FULL"

## 2. Setup

Walks up from the working directory until it finds the repo (`config.yaml` + `src/bdb27`), puts
`src/` on the path, and imports `bdb27.fallback_validation` — **the same module the headless
runner uses**. The estimators, the label rules, the rendering and the artefact writer exist in
exactly one place, so this notebook cannot drift from `scripts/fallback_validation.py`.

In [ ]:
import sys
from pathlib import Path


def find_repo_root() -> Path:
    '''Walk upward from the working directory until the repo is found.'''
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "config.yaml").exists() and (parent / "src" / "bdb27").is_dir():
            return parent
    raise RuntimeError(
        "Cannot locate the BDB27 repo root (need config.yaml + src/bdb27). "
        "Open this notebook from inside the BDB27 checkout.")


REPO = find_repo_root()
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

import pandas as pd  # noqa: E402

from bdb27 import fallback_validation as fv  # noqa: E402

print(f"repo: {REPO}")
print(f"module: {fv.__file__}")
print(f"spec: {fv.SPEC}   seed: {fv.SEED}   bootstrap reps: {N_BOOTSTRAP}")

In [ ]:
# ---------------------------------------------------------------- resolve inputs
# The ranking prefers the FULL run's folder and the D190 floor2.5 control pass, then
# an exact filename, then the shallowest hit — and every candidate is printed, so a
# sample copy carrying the same filename is visible rather than silently opened.
roots = fv.search_roots(REPO)
PREFER = {
    "game_turns": ("full_run_20261111", "data/interim"),
    "combine_turns": ("floor2.5_control", "speed_floor"),
    "combine_results": ("data/raw",),
}
found = {
    "game_turns": fv.resolve_input(
        GAME_TURNS, label="GAME_TURNS",
        patterns=("bdb27_full_db_turns.parquet", "*full_db_turns*.parquet"),
        roots=roots, prefer=PREFER["game_turns"],
        hint="set GAME_TURNS in §1 to the full path of bdb27_full_db_turns.parquet"),
    "combine_turns": fv.resolve_input(
        COMBINE_TURNS, label="COMBINE_TURNS",
        patterns=("combine_db_turns_floor2.5_control.csv", "combine_db_turns*.csv"),
        roots=roots, prefer=PREFER["combine_turns"],
        hint="set COMBINE_TURNS in §1 to outputs/sensitivity/speed_floor/"
             "combine_db_turns_floor2.5_control.csv"),
    "combine_results": fv.resolve_input(
        COMBINE_RESULTS, label="COMBINE_RESULTS", patterns=("combine_results.csv",),
        roots=roots, prefer=PREFER["combine_results"],
        hint="set COMBINE_RESULTS in §1 to the full path of combine_results.csv"),
}
game_path = found["game_turns"]["chosen"]
combine_turns_path = found["combine_turns"]["chosen"]
results_path = found["combine_results"]["chosen"]
out_dir = Path(OUT_DIR) if OUT_DIR else (REPO / "outputs" / "fallback_validation")

for label, hit in found.items():
    others = [c for c in hit["candidates"] if c != str(hit["chosen"])]
    print(f"{label:<16}: {hit['chosen']}")
    print(f"{'':<16}  chosen by {hit['how']}; "
          f"{len(hit['candidates'])} candidate(s) matched")
    for other in others[:5]:
        print(f"{'':<16}  also matched: {other}")
print(f"{'out dir':<16}: {out_dir}")
print("no tracking file is read by this notebook; no position group other than DB is used")

## 3. Leg 1 — reliability of the 30–60° band (spec §1)

**Estimand.** The intraclass correlation of `apex_retention` across turns within the same player,
at 30 ≤ `abs_turn_angle` < 60 in games. How much of the between-player spread is signal about the
player rather than noise between his own turns.

**Sample.** `setting == 'game'`, `position_group == 'DB'`, `valid`, band turns only. No opponent
identifier, no `player_play` merge, no tracking file.

**Three estimators, all primary, all printed.** ICC(1) one-way random effects on ANOVA moments with
`k0` for the unbalanced design and a player-level bootstrap interval (2,000 resamples, seed
20261115, percentile); split-half odd/even by `(season, game_id, play_id, frame_start)` with
Spearman–Brown; and the 2024↔2025 correlation of per-player season means, printed for both n
variants. `R_mean` follows as a consequence — it is the quantity that attenuates leg 2.

**Display rule (spec §7).** Where `n` observations ≤ `n` parameters the interval prints as
`not estimable (saturated)` with the cell's counts. The point estimate is still printed and the
estimator's maths is unchanged.

In [ ]:
game = fv.load_game_turns(game_path)
print(f"game turns loaded: {len(game):,} valid DB turns in 30-180 deg {STAMP}")

rel = fv.reliability(game)
print()
print("\n".join(fv.render_reliability(rel)))

## 4. Leg 2 — discriminant validity of the battery (spec §2)

**Estimand.** For each drill in {3-cone, short shuttle, 40-yard, 10-yard split}: how strongly it
predicts a player's **shallow-break** retention, against how strongly it predicts his **deep-plant**
retention, on the same players.

* **Outcome S** — mean `apex_retention` over 30–60° (the band under test).
* **Outcome P** — mean `apex_retention` over 120–150°, the band the battery over-supplies. 150–180°
  is carried as a declared sensitivity, never as a third primary outcome.
* **Predictor** — the Combine drill, sign-flipped so positive = faster, z-scored on its own
  estimation sample. A coefficient reads *retention change per one SD of the drill*.
* **Primary model** — univariate OLS on the player means with HC1 standard errors. Univariate on
  purpose: with 23 players on the 3-cone a four-drill joint model is not identified in any useful
  sense, and the drills are collinear by construction.
* **The discriminant statistic** — Δ = b_P − b_S, by a **paired** player-level bootstrap that refits
  both equations on the same resampled players. One significant and one not is not a test
  (Gelman & Stern 2006); Δ with its own interval is.
* **Materiality bar, fixed before the fit** — a quarter of the between-player SD of the outcome:
  `|b_S| ≥ 0.25·SD(S)` and `|b_P| ≥ 0.25·SD(P)`.
* **MDE** — printed and labelled post-hoc, so a null here is read "not detectable", never "not
  present".

**The constraint that governs the whole leg** is join coverage: the two drills that contain a
turning test have 23 and 31 players of the 115 game DBs. It is pre-declared, printed with every
coefficient, and no drill is dropped for being small.

In [ ]:
combine_results = fv.load_combine_results(results_path)
print(f"Combine DB drill rows: {len(combine_results)} {STAMP}")

discriminant, disc_meta = fv.discriminant(game, combine_results, reps=N_BOOTSTRAP)
print()
print("\n".join(fv.render_discriminant(discriminant, disc_meta)))

## 5. Leg 3 — speed coverage (spec §3)

Descriptive. **No criterion, no verdict, no inference** — this leg exists because it needs none.
What entry speed does the Combine ever record, and what does the 30–60° band actually require?

The single number the leg exists for: the **maximum entry speed observed anywhere in the Combine DB
set**, and the share of game band turns whose entry speed exceeds it. That is the gap stated as a
count and a share, with no model in it. The crosstab then shows what the marginals hide — the
Combine records high entry speeds, and it records shallow angles, but whether it records them
**together** is a question of counts.

**No figure is produced.** `TASK.md` allows exactly four and forbids a fifth.

In [ ]:
combine = fv.load_combine_turns(combine_turns_path)
print(f"Combine DB turns: {len(combine):,} across "
      f"{combine['drill_name'].nunique()} drills {STAMP}")

speed = fv.speed_coverage(game, combine)
print()
print("\n".join(fv.render_speed(speed)))

## 6. The verdict (spec §4) and the five artefacts

The rules are evaluated in the order written in the spec, before the fit. The first that fires gives
the headline; every later rule still prints, so the reader can see what *would* have been said.

The artefacts are written by the same function the headless runner uses, so a Run All here produces
byte-identical files to `scripts/fallback_validation.py` — same seed, same order, same rounding.

| File | Contents |
|---|---|
| `reliability.json` | leg 1 in full: ICC, both intervals, split-half, seasons, `R_mean`, label |
| `discriminant.csv` | leg 2: one row per drill, S and P fits, Δ, bars, MDEs, weighted sensitivity |
| `speed_coverage.json` | leg 3: percentiles, the 26 values, the gap, the crosstab |
| `verdict.txt` | the headline, the rule that fired, and every rule that did not |
| `manifest.json` | sha256 of every input and output, git HEAD, seed, config |

In [ ]:
bundle = fv.assemble(rel, discriminant, disc_meta, speed, reps=N_BOOTSTRAP)
manifest = fv.write_outputs(out_dir, bundle, inputs={
    "game_turns": game_path,
    "combine_turns": combine_turns_path,
    "combine_results": results_path,
}, repo_root=REPO)

print(bundle.text)
print(f"written to {out_dir}:")
for name, digest in manifest["outputs"].items():
    print(f"  {name:<20} sha256 {digest[:16]}…")
print(f"manifest: git HEAD {manifest['git_head']}, seed {manifest['seed']}, "
      f"reps {manifest['bootstrap_reps']}, verdict {manifest['verdict']['label']}")
print()
print("This notebook does not upgrade a null into a claim, and it does not stop on a number.")
print("A wide interval on 23 players is an absence of evidence, and it is reported as such.")